# k-XOR instance-loading oracles

Complete native counterpart of Qualtran v0.7.0 `load_kxor_instance.ipynb`,
revision `bcb17bb25360ed68ce3bc084645db3e929799389`, frozen blob
`9a08818e673ccc5621e4a077ca1fab1f840a4075`.

The three declared oracles load a unique scope, declare its inverse index query,
and declare its normalized RHS rotation. This notebook follows the original
scope-loader examples, signatures and symbolic one-level call graph. Scope
indices start at zero and pack in big-endian order. Output scope wires start
clean and belong to the caller. Source index/RHS roots carry declaration-only
execution boundaries recorded in `KXOR_LOADING_FILE_AUDIT.md`.

Copyright 2024 Google LLC. Adapted under Apache-2.0. Source references
*Quartic quantum speedups for planted inference*, arXiv:2406.19378v1,
Notation 2.24 and Theorem 4.17.


In [ ]:
from collections.abc import Callable
from typing import cast

from IPython import display as notebook_display

from flagquantum.algorithms.kxor_loading import (
    example_scope_loader,
    symbolic_scope_loader,
)
from flagquantum.circuit import Circuit
from flagquantum.core.ir.program.signatures import Signature
from flagquantum.drawer.call_graph import export_call_graph_dot, render_call_graph_svg
from flagquantum.drawer.signature_graph import export_signature_graph_dot

SVG = cast(Callable[[bytes], object], notebook_display.SVG)
display = cast(Callable[[object], None], notebook_display.display)


def show_signature(name: str, signature: Signature) -> None:
    if all(type(register.bitsize) is int for register in signature):
        dot = export_signature_graph_dot(signature, name)
    else:
        rows = {}
        for register in signature:
            rows[register.name + " width"] = str(register.bitsize)
            rows[register.name + " side"] = str(register.side.name)
            rows[register.name + " encoding"] = str(register.dtype.encoding)
            rows.update({key: str(value) for key, value in register.dtype.parameters})
        dot = export_call_graph_dot(name, {}, annotations={name: rows})
    display(SVG(render_call_graph_svg(dot)))


## `LoadConstraintScopes`
Given an index $j$, load the scope of the $j$-th unique constraint.

Given a $k$-XOR-SAT instance `inst` with $n$ variables and $m$ constraints.
Assuming `inst` has $\bar{m}$ unique constraints, we define $U_j \in {[n] \choose k}$
for $j \in [\bar{m}]$ as the $j$-th unique constraint scope.

See :class:`KXorInstance` for the overall problem definition.

The scopes are loaded using a QROM. If the input contains an invalid index, then
the output can be an arbitrary value.

#### Registers
 - `j`: a number in [\bar{m}]
 - `U`: $j$-th unique scope
 - `ancilla`: entangled intermediate qubits, to be uncomputed by the adjoint. 

#### References
 - [Quartic quantum speedups for planted inference](https://arxiv.org/abs/2406.19378v1). Theorem 4.17, proof para 2.


In [ ]:
# The complete loader belongs to the native Algorithms module imported above.


### Example Instances

In [ ]:
load_scopes_symb = symbolic_scope_loader()


In [ ]:
load_scopes = example_scope_loader()


#### Graphical Signature

In [ ]:
show_signature("load_scopes_symb", load_scopes_symb.signature)
show_signature("load_scopes", load_scopes.signature)


### Call Graph

In [ ]:
load_scopes_symb_graph = load_scopes_symb.source_graph(
    root="LoadConstraintScopes", lookup="QROM"
)
display(
    SVG(
        render_call_graph_svg(
            export_call_graph_dot(
                load_scopes_symb_graph.root, load_scopes_symb_graph.direct_calls
            )
        )
    )
)
load_scopes_symb_sigma = load_scopes_symb_graph.leaf_counts
display(load_scopes_symb_sigma)
load_scopes_symb_model = load_scopes_symb.qrom.resource_formula()
display(load_scopes_symb_model)


In [ ]:
# The original complete physical example uses the shared QROM body.
operation = load_scopes.make_on(
    (0, 1, 2),
    tuple(range(3, 15)),
    workspace=(15, 16),
    classical_bits=(90, 91),
)
circuit = Circuit(17).append(operation)
primitive_count = len(circuit.to_ir().instructions)
measurement_count = sum(gate.name == "measure" for gate in operation.instructions)
assert primitive_count == 86 and measurement_count == 3
model_categories = load_scopes.qrom.resource_model().gates
assert model_categories.and_bloq == 3 and model_categories.measurement == 3
# The 86 emitted instructions and these QEC categories are separate quantities.
display({"emitted_primitives": primitive_count, "measurements": measurement_count})
display(model_categories)
